In [ ]:
# Move kaggle.json to the right place
import os
from google.colab import files # Import files module

os.makedirs("/root/.kaggle", exist_ok=True)

# Check if kaggle.json exists. If not, prompt user to upload.
if not os.path.exists("kaggle.json"):
    print("kaggle.json not found. Please upload your kaggle.json file.")
    files.upload() # This will prompt the user

# Now, attempt to move the file
# Check again after potential upload
if os.path.exists("kaggle.json"):
    os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 600)
else:
    print("kaggle.json still not found after attempting upload. Please ensure you upload the correct file.")

# Install kaggle library
!pip install kaggle -q

# Download the dataset
# This part might fail if kaggle.json was not successfully moved and configured.
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge

# Unzip it
!unzip -q jigsaw-toxic-comment-classification-challenge.zip -d data/
!ls data/

kaggle.json not found. Please upload your kaggle.json file.


Saving kaggle.json to kaggle.json
100% 52.6M/52.6M [00:00<00:00, 58.8MB/s]

sample_submission.csv.zip  test.csv.zip  test_labels.csv.zip  train.csv.zip


In [ ]:
import zipfile, os

# Unzip all zip files inside the data folder
for file in os.listdir("data"):
    if file.endswith(".zip"):
        with zipfile.ZipFile(f"data/{file}", "r") as z:
            z.extractall("data/")
            print(f"Extracted: {file}")

print("\nData folder now:")
print(os.listdir("data"))

Extracted: test.csv.zip
Extracted: test_labels.csv.zip
Extracted: train.csv.zip
Extracted: sample_submission.csv.zip

Data folder now:
['sample_submission.csv', 'train.csv', 'test.csv.zip', 'test_labels.csv.zip', 'train.csv.zip', 'sample_submission.csv.zip', 'test.csv', 'test_labels.csv']


In [ ]:
import pandas as pd

df = pd.read_csv("data/train.csv")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
print("\nLabel counts:")
print(df[label_cols].sum())
print("\nLabel percentages:")
print((df[label_cols].mean() * 100).round(2))

Shape: (159571, 8)

First 5 rows:
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3             0        0       0       0              0  
4             0        0       0       0              0  

Label counts:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
i

In [ ]:
import pandas as pd

df = pd.read_csv("data/train.csv")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
print("\nLabel counts:")
print(df[label_cols].sum())
print("\nLabel percentages:")
print((df[label_cols].mean() * 100).round(2))

print("\nAny missing values?")
print(df.isnull().sum())

Shape: (159571, 8)

First 5 rows:
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3             0        0       0       0              0  
4             0        0       0       0              0  

Label counts:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
i

In [ ]:
!pip install emoji indic-nlp-library -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 11.7 MB/s eta 0:00:00


In [ ]:
import re
import emoji

def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"<.*?>", "", text)              # remove HTML tags
    text = re.sub(r"\s+", " ", text).strip()       # fix whitespace
    return text

def handle_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

def normalize_repeated_chars(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)    # soooo -> so

def normalize_abbreviations(text):
    abbrevs = {
        "wtf": "what the hell", "stfu": "shut up",
        "idk": "i don't know", "ngl": "not gonna lie",
        "lmao": "laughing", "omg": "oh my god",
        "u": "you", "ur": "your", "r": "are",
        "bc": "because", "2": "to", "4": "for"
    }
    tokens = text.lower().split()
    tokens = [abbrevs.get(t, t) for t in tokens]
    return " ".join(tokens)

def full_pipeline(text):
    text = clean_text(text)
    text = handle_emojis(text)
    text = normalize_repeated_chars(text)
    text = normalize_abbreviations(text)
    return text.lower().strip()

# Test it on a few samples
test_samples = [
    "Check this out http://example.com 😡😡 ur sooooo stupid wtf",
    "yaar kya scene hai bro 🔥🔥🔥",
    "stfu u idiot lmaooo",
]

for s in test_samples:
    print("BEFORE:", s)
    print("AFTER: ", full_pipeline(s))
    print()

BEFORE: Check this out http://example.com 😡😡 ur sooooo stupid wtf
AFTER:  check this out enraged_face enraged_face your soo stupid what the hell

BEFORE: yaar kya scene hai bro 🔥🔥🔥
AFTER:  yaar kya scene hai bro fire fire fire

BEFORE: stfu u idiot lmaooo
AFTER:  shut up you idiot lmaoo



In [ ]:
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

print(f"CPU cores available: {cpu_count()}")

with Pool(cpu_count()) as pool:
    clean_texts = list(tqdm(
        pool.imap(full_pipeline, df["comment_text"].tolist()),
        total=len(df),
        desc="Preprocessing"
    ))

df["clean_text"] = clean_texts

print(df[["comment_text", "clean_text"]].head(10))
print("\nDone! Total rows processed:", len(df))

CPU cores available: 2


Preprocessing: 100%|██████████| 159571/159571 [01:48<00:00, 1464.74it/s]


                                        comment_text  \
0  Explanation\nWhy the edits made under my usern...   
1  D'aww! He matches this background colour I'm s...   
2  Hey man, I'm really not trying to edit war. It...   
3  "\nMore\nI can't make any real suggestions on ...   
4  You, sir, are my hero. Any chance you remember...   
5  "\n\nCongratulations from me as well, use the ...   
6       COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK   
7  Your vandalism to the Matt Shirvington article...   
8  Sorry if the word 'nonsense' was offensive to ...   
9  alignment on this subject and which are contra...   

                                          clean_text  
0  explanation why the edits made under my userna...  
1  d'aww! he matches this background colour i'm s...  
2  hey man, i'm really not trying to edit war. it...  
3  " more i can't make any real suggestions on im...  
4  you, sir, are my hero. any chance you remember...  
5  " congratulations from me as well, use the too... 

In [ ]:
from sklearn.model_selection import train_test_split

# Create a binary label — 1 if ANY label is toxic, 0 if clean
df["label"] = (df[["toxic", "severe_toxic", "obscene",
                    "threat", "insult", "identity_hate"]].sum(axis=1) > 0).astype(int)

print("Label distribution:")
print(df["label"].value_counts())
print("\nPercentage:")
print((df["label"].value_counts(normalize=True) * 100).round(2))

# Split
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])

print(f"\nTrain size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")

Label distribution:
label
0    143346
1     16225
Name: count, dtype: int64

Percentage:
label
0    89.83
1    10.17
Name: proportion, dtype: float64

Train size: 143613
Val size:   15958


In [ ]:
!pip install transformers torch -q

import torch
from transformers import AutoTokenizer

MODEL_NAME = "google/muril-base-cased"

print("Loading MuRIL tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test the tokenizer
sample = "yaar tu bahut bura hai bro"
tokens = tokenizer(sample, return_tensors="pt")
print("\nSample text:", sample)
print("Token IDs:", tokens["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))
print("\nTokenizer working!")

Loading MuRIL tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]


Sample text: yaar tu bahut bura hai bro
Token IDs: tensor([[  104, 51206, 23500,  5208, 67440,  1297, 51576,  1323,   105]])
Tokens: ['[CLS]', 'yaar', 'tu', 'bahut', 'bura', 'hai', 'br', '##o', '[SEP]']

Tokenizer working!


In [ ]:
from torch.utils.data import Dataset

class AbuseDataset(Dataset):
    def __init__(self, texts, labels, max_len=128):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.float)
        }

# Create datasets
train_dataset = AbuseDataset(
    train_df["clean_text"].tolist(),
    train_df["label"].tolist()
)
val_dataset = AbuseDataset(
    val_df["clean_text"].tolist(),
    val_df["label"].tolist()
)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:  ", len(val_dataset))
print("\nSample item keys:", train_dataset[0].keys())

Train dataset size: 143613
Val dataset size:   15958

Sample item keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
from transformers import AutoModelForSequenceClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1  # binary classification
)
model = model.to(DEVICE)

print("Model loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Using device: cuda


pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

Model loaded!
Parameters: 237,556,993


In [ ]:
import os
from google.colab import userdata
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from tqdm import tqdm
import wandb
import torch

# ---- W&B Setup via Colab Secrets ----
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

wandb.init(
    project="multilingual-abuse-detection",
    name="muril-binary-v1",
    config={
        "model":      MODEL_NAME,
        "batch_size": 32,       # increased from 16 — faster on GPU
        "epochs":     3,
        "lr":         2e-5,
        "max_len":    128,
        "optimizer":  "AdamW",
        "dataset":    "kaggle-toxic-comments"
    }
)

# Config
BATCH_SIZE = 32       # bumped up for GPU efficiency
EPOCHS     = 3
LR         = 2e-5

# Dataloaders — pin_memory + num_workers speeds up data loading
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          pin_memory=True, num_workers=2)

# Class weights to handle imbalance
pos = train_df["label"].sum()
neg = len(train_df) - pos
pos_weight = torch.tensor([neg / pos]).to(DEVICE)
print(f"Class weight (positive): {pos_weight.item():.2f}")

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=500,
    num_training_steps=total_steps
)

# ---- Mixed Precision Scaler (2-3x faster on T4 GPU) ----
scaler = torch.cuda.amp.GradScaler()

# Training loop
best_f1 = 0

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    total_loss = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=True)

    for i, batch in enumerate(train_bar):
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels         = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        # Mixed precision forward pass
        with torch.cuda.amp.autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits.squeeze(), labels)

        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

        # Update tqdm bar with live loss
        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        if i % 100 == 0:
            wandb.log({"step_loss": loss.item(), "step": epoch * len(train_loader) + i})

    avg_loss = total_loss / len(train_loader)

    # --- Validate ---
    model.eval()
    all_preds, all_labels = [], []

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  ", leave=True)

    with torch.no_grad():
        for batch in val_bar:
            input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
            attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels         = batch["labels"].to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            probs = torch.sigmoid(outputs.logits.squeeze())
            preds = (probs > 0.5).int()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.int().cpu().numpy())

            val_bar.set_postfix({"samples": len(all_preds)})

    val_f1 = f1_score(all_labels, all_preds, average="macro")

    wandb.log({
        "epoch":      epoch + 1,
        "train_loss": avg_loss,
        "val_f1":     val_f1,
    })

    print(f"\n✅ Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}\n")

    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained("/content/best_model")
        tokenizer.save_pretrained("/content/best_model")
        print(f"  💾 Best model saved! (F1: {best_f1:.4f})\n")

wandb.finish()
print("Training complete! Best F1:", best_f1)

step,▁▃▆█
step_loss,██▇▁
step,300
step_loss,0.82154


/tmp/ipykernel_3774/278094739.py:55: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Class weight (positive): 8.84


Epoch 1/3 [Train]:   0%|          | 0/4488 [00:00<?, ?it/s]/tmp/ipykernel_3774/278094739.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/3 [Val]  :   0%|          | 0/499 [00:00<?, ?it/s]/tmp/ipykernel_3774/278094739.py:109: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/3 [Val]  : 100%|██████████| 499/499 [00:28<00:00, 17.55it/s, samples=15958]



✅ Epoch 1/3 | Train Loss: 0.5486 | Val F1: 0.8976



...


✅ Epoch 3/3 | Train Loss: 0.3256 | Val F1: 0.9056



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  💾 Best model saved! (F1: 0.9056)



epoch,▁▅█
step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇████
step_loss,▂▂▂▂▂▂▁▁▂▁▁▃▁█▂▁▂▁▁▁▁▃▂▂▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁
train_loss,█▃▁
val_f1,▁▅█
epoch,3
step,13376
step_loss,0.00418
train_loss,0.32561
val_f1,0.90562


Training complete! Best F1: 0.9056207579562907
